In [ ]:
# the codes in this cell are not important

from multiprocessing import context
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

# calculate the attention scores for the second input

query = inputs[1]  # 2nd input token is the query

attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query) # dot product (transpose not necessary here since they are 1-dim vectors)

print(attn_scores_2)


# calculate the attention weights for the second input
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)

print("Attention weights:", attn_weights_2)
print("Sum:", attn_weights_2.sum())


# calculate the context vector for the second input

query = inputs[1] # 2nd input token is the query

context_vec_2 = torch.zeros(query.shape) 
for i,x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i]*x_i

print(context_vec_2)


## explain the following lines of code

"""

1st line: attn_scores_2 = torch.empty(inputs.shape[0])

This line creates an empty tensor of shape (inputs.shape[0],) to store the attention scores for each input token.

2nd line: context_vec_2 = torch.zeros(query.shape) 

This line creates a tensor of shape (query.shape) to store the context vector for the second input.


"""

## for ALL INPUTS

attn_scores = inputs @ inputs.T

attn_weights = torch.softmax(attn_scores, dim=-1) # what is the last dimension? the last dimension is the number of input tokens.

context_vec = attn_weights @ inputs



In [ ]:
## self-attention with trainable weights
import torch.nn as nn

class SelfAttention_v2(nn.Module):

    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))